# Step 70 — Network history, converted-channel audit and StationXML

Run **after the BOB steps (60–69)**, so that StationXML is built only once every dataset is on UTC. It reads the converted waveform archive (step 10) and the station evidence. This notebook separates three things: documented equipment/deployment history, channel identities actually present in the converted waveform archive, and a response model. None is a substitute for the other.

**Purpose:** refresh header evidence from the exact configured output archive; reconstruct the documentary network; expose metadata conflicts; create traceable draft StationXML; audit epoch selection and response completeness. Step 70 is the post-conversion review and packaging step.

**What running this notebook does:** reads waveform headers and historical reference files; writes tables, maps, diagrams, XML and provenance beneath `conversion_config.json`'s `output_root/FAIR/metadata/network_history`. It does not edit legacy inputs, change waveform samples, correct clocks or relocate stations. Full archive header scanning can take several minutes. It replaces generated network-history outputs in that directory; preserve an earlier run externally if needed.

**Scientific status:** provisional metadata. Schema-valid XML is not proof of correct instruments, calibration or timing. A complete counts-to-ground-motion response remains an explicit target; missing electronics must not be replaced with invented unity stages.


**Code organization:** the active reconstruction, header scan, response-model, plotting and provenance functions are embedded below. Shared configuration and third-party libraries remain imported. Reference files stay in `LEGACY/STATION_METADATA/NETWORK_HISTORY`; the ZIP is a supplied snapshot, not a runtime dependency. After editing definitions, rerun them before the execution cells. A full Run All regenerates external outputs and may rescan the complete waveform archive.


## Workflow guide (formerly network_history/README.md)

No network-history code-support folder is required. All reconstruction and metadata-audit functions are in this notebook; validation checks are in `71_validate_network_history.ipynb`. Shared configuration and library utilities remain imported.

| Material | Canonical location |
|---|---|
| Frozen historical references and curated evidence | `P['NETWORK_HISTORY_INPUT']`: normally `LEGACY/STATION_METADATA/NETWORK_HISTORY` |
| Original station files | `LEGACY/STATION_METADATA/PIN_STA.DAT`, `PIN_STA_ALL.txt`, `STATIONS.INP` |
| BOB station descriptions | `LEGACY/BOB/STATIONS.TXT` |
| Generated tables, figures, header audits and StationXML | `P['NETWORK_HISTORY_OUTPUT']`: normally `<output_root>/FAIR/metadata/network_history` |
| Validation reports | `NETWORK_HISTORY_OUTPUT/validation` (step 71) |
| Optional Antelope metadata export | Step 82, using the StationXML from the configured FAIR path |

Run **10 → 15 → 16** to convert waveforms, reconstruct metadata and validate. Steps 80–82 (SEISAN and CSS3.0 exports) run after this step. Step 82's inventory CSS export omits response objects and cannot preserve all sub-day epoch detail; retain StationXML as the full metadata reference.

The pipeline does not load station/reference metadata from the code repository. Canonical legacy station files are hashed and referenced, not duplicated into the curated reference folder. Generated scan/cache/provenance products belong in FAIR. Earlier migration audits record checksum verification and retained external backups; historical product snapshots are not active metadata inputs.

The conversion tests (`tests/test_network_reconstruction.py`, through step 71) load only the tagged definition cells. They do not need the former network_history Python package or its README. Keep the LEGACY reference directory and FAIR products: removing the obsolete repository folder does not remove those data.


## Input and generated-metadata storage

Reference inputs are read from `P['NETWORK_HISTORY_INPUT']`, defaulting to `LEGACY/STATION_METADATA/NETWORK_HISTORY`. This is a frozen documentary/reference bundle, including explicitly identified modern contextual inputs and curated historical interpretations; it is not all original 1991 measurement data. Preserve source hashes and origin notes.

All generated network metadata are written to `P['NETWORK_HISTORY_OUTPUT'] = FAIR/metadata/network_history` beneath the configured output root: source-derived station/epoch tables, alternative coordinates, header scans/audits, response assumptions copied for provenance, figures, draft XML and validation manifests. Header-derived metadata are never read from repository data or silently substituted for documentary inputs.

The code repository keeps the notebook, configuration and tests, not metadata or relief/header-scan files. Old repository-generated tables are archived under the FAIR network-history `legacy_pipeline_snapshots` folder; they are historical products, not the active scan cache. The previous external network-history output folder is retained as a migration backup, while configured consumers use the new FAIR path.


### Canonical station files: do not duplicate

`PIN_STA.DAT`, `PIN_STA_ALL.txt` and `STATIONS.INP` are read/referenced at the top of `LEGACY/STATION_METADATA`, not copied into NETWORK_HISTORY. The BOB station-description file remains at `LEGACY/BOB/STATIONS.TXT`. All four previously copied NETWORK_HISTORY files were confirmed byte-identical by SHA-256, including the uploaded ZIP versions. Curated evidence JSONs refer to those original documents; they are distinct interpretation products, not replacement station files.

`CANONICAL_REFERENCE_FILES` makes the canonical paths explicit. Provenance packaging hashes them and writes a path/hash manifest in FAIR without copying the raw station files again. Frozen source bundles from earlier runs may retain historical copies; these are not active input locations. Do not recreate duplicates to make an old local path work.


## 1. Load the shared configuration and display the actual paths

This block locates `10_conversion` from either the repository root or notebook folder, imports the pipeline path resolver, and identifies both the configured **converted WAV database** and external output directory. It deliberately does not reconstruct a waveform path from `DATA_TOP/FAIR`: that would select the older assembled archive rather than the current conversion run. Restart the kernel after updating local modules.


In [ ]:
from pathlib import Path
import sys,json,hashlib,shutil
from datetime import datetime,timezone
from IPython.display import display,Image,Markdown
import pandas as pd
from obspy import read_inventory,UTCDateTime
candidates=[Path.cwd(),Path.cwd()/'10_conversion',*Path.cwd().parents]
PIPELINE=next((p for p in candidates if (p / 'lib' / 'clock.py').is_file() and (p/'70_network_history_and_stationxml.ipynb').is_file()),None)
if PIPELINE is None:raise FileNotFoundError('Open from the project root or 10_conversion folder')
sys.path.insert(0,str(PIPELINE.resolve()))
from lib.config import pipeline_paths
CONFIG_FILE=None
P=pipeline_paths(config_file=CONFIG_FILE)
DATA=Path(P['NETWORK_HISTORY_INPUT'])
BASE=Path(PIPELINE)
if not DATA.is_dir():raise FileNotFoundError('Missing LEGACY reference metadata: '+str(DATA))
WAV_ROOT=Path(P['SEISAN_WAV_DB'])
OUTPUT=Path(P['NETWORK_HISTORY_OUTPUT'])
HEADER_OUTPUT=OUTPUT/'header_scan'
print('Configuration:',P['CONFIG_FILE'])
print('Converted waveforms:',WAV_ROOT)
print('Generated network metadata:',OUTPUT)
if not WAV_ROOT.is_dir():raise FileNotFoundError('Run step 10 first or check conversion_config.json: '+str(WAV_ROOT))

## Embedded reconstruction code: evidence and channel epochs

All active network-history workflow code is now defined in this notebook. The `LEGACY/STATION_METADATA/NETWORK_HISTORY` folder remains read-only reference material: source catalogues, historical evidence, coordinate alternatives, response assumptions and the modern relief grid. It is not another executable pipeline.

The following definitions load those references and build channel epochs. `build_channel_epochs()` builds every epoch on UTC. Documentary pre-eruption epochs come from the field notes, whose dates and times are Philippine local time (UTC+8): a date-only bound is local midnight, 16:00 UTC the day before, and ends are exclusive. Post-eruption identities (those recorded in July/August) start at their first surviving header after the climax; CAB/BUG Z continue from the end of their documentary epochs (15 June, local midnight). The converted archive (step 10, already on UTC) then decides the rest: data labelled before a documented installation extend that epoch's start, and data still labelled after a documented end get their own `label_after_documented_end` epoch, with an unknown sensor. The build fails if any archived trace falls outside every epoch of its identity. Epoch sample rates are the archive's 3125/31 Hz. None of this proves uninterrupted operation or instrument continuity.

Date-only starts/ends and placeholder depths are inherited reconstruction choices, not newly measured values. Undated PI2L and auxiliary tilt entries remain provisional. The function name deliberately differs from the displayed `epochs` table so that repeated runs do not shadow executable code. No sources or waveforms are changed by defining these functions.

In [ ]:
from pathlib import Path
from lib.config import pipeline_paths
BASE=Path(PIPELINE)
_NETWORK_PATHS=pipeline_paths(config_file=globals().get('CONFIG_FILE'))
DATA=Path(_NETWORK_PATHS['NETWORK_HISTORY_INPUT'])
DEFAULT_HEADER_DATA=Path(_NETWORK_PATHS['NETWORK_HISTORY_OUTPUT'])/'header_scan'
CANONICAL_REFERENCE_FILES = {
    'PIN_STA.DAT': Path(_NETWORK_PATHS['LEGACY_TOP'])/'STATION_METADATA/PIN_STA.DAT',
    'PIN_STA_ALL.txt': Path(_NETWORK_PATHS['LEGACY_TOP'])/'STATION_METADATA/PIN_STA_ALL.txt',
    'STATIONS.INP': Path(_NETWORK_PATHS['LEGACY_TOP'])/'STATION_METADATA/STATIONS.INP',
    'BOB_STATIONS.TXT': Path(_NETWORK_PATHS['LEGACY_TOP'])/'BOB/STATIONS.TXT',
}

from pathlib import Path

import os, json, copy, csv, re, tempfile, sys

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

from obspy import UTCDateTime, read_inventory

from obspy.core.inventory import Inventory, Network, Station, Channel, Site, Comment, Equipment

from obspy.core.inventory.response import Response, PolesZerosResponseStage, ResponseStage, InstrumentSensitivity, CoefficientsTypeResponseStage

def load(name):
    return json.loads((DATA / name).read_text())

# Archive sample rate: the DMX header rate 100.80645 Hz is the float32 image of 3125/31 Hz (step 10).
SEISMIC_RATE_HZ = 3125 / 31
PHT_OFFSET = pd.Timedelta(hours=8)  # Philippine time = UTC + 8 h


def pht_to_utc(local):
    """UTC ISO string for a Philippine local date or date-time from the field notes.

    A date-only bound is local midnight, i.e. 16:00 UTC the day before. Ends are exclusive:
    an end date of 16 June means the epoch runs to the start of 16 June, local time.
    """
    if local is None:
        return None
    return (pd.Timestamp(local) - PHT_OFFSET).strftime('%Y-%m-%dT%H:%M:%S')


def observed_identity_bounds(headers):
    """First and last UTC sample time of every station/channel in the converted archive."""
    h = headers.assign(first=pd.to_datetime(headers['first'], utc=True).dt.tz_localize(None),
                       last=pd.to_datetime(headers['last'], utc=True).dt.tz_localize(None))
    return h.groupby(['station', 'channel']).agg(first=('first', 'min'), last=('last', 'max'), days=('date', 'nunique')).reset_index()


def uncovered_data(rows, headers):
    """Daily header rows (UTC) whose data fall outside every epoch of their station/channel."""
    out = []
    h = headers.assign(first=pd.to_datetime(headers['first'], utc=True).dt.tz_localize(None),
                       last=pd.to_datetime(headers['last'], utc=True).dt.tz_localize(None))
    for r in h.itertuples(index=False):
        spans = sorted((pd.Timestamp(e['start']) if e['start'] else pd.Timestamp.min,
                        pd.Timestamp(e['end']) if e['end'] else pd.Timestamp.max)
                       for e in rows if (e['station'], e['channel']) == (r.station, r.channel))
        merged = []  # adjacent epochs (end == next start) cover a day that straddles their boundary
        for a, b in spans:
            if merged and a <= merged[-1][1]:
                merged[-1][1] = max(merged[-1][1], b)
            else:
                merged.append([a, b])
        if not any(a <= r.first and r.last <= b for a, b in merged):
            out.append(dict(station=r.station, channel=r.channel, date=r.date, first=r.first, last=r.last, files=r.files))
    return pd.DataFrame(out, columns=['station', 'channel', 'date', 'first', 'last', 'files'])


def build_channel_epochs(header_data_dir=None):
    """Channel epochs on UTC.

    Documentary epochs come from the field notes (Lockhart Appendix 1 and chronology), whose dates and
    times are Philippine local time; they are converted to UTC here. The converted archive (step 10,
    already on UTC) then decides the rest: post-eruption identities start at their first surviving
    header, data labelled before a documented installation extend that epoch's start, and data labelled
    after a documented end get their own epoch, so every archived trace falls inside an epoch.
    """
    rows = []
    local_note = 'Field-note dates are Philippine local (UTC+8), converted to UTC; a date-only bound is local midnight, and ends are exclusive.'

    def add(sta, start, end=None, sensor='L4-C', att=None, ch='EHZ', depth=1.0, note='', rate=SEISMIC_RATE_HZ, *, utc=False, basis='documentary'):
        if not utc:
            start, end = pht_to_utc(start), pht_to_utc(end)
            note = (note + ' ' + local_note).strip()
        rows.append(dict(station=sta, channel=ch, start=start, end=end, sensor=sensor, attenuation_db=att, depth_m=depth,
                         sample_rate_hz=rate, basis=basis, note=note))
    add('LIL', '1991-04-25', '1991-05-01', att=30)
    add('PIE', '1991-04-30', '1991-06-16', att=18, note='Destroyed June 15; the epoch ends with that local day, not at a measured failure time.')
    add('UBO', '1991-05-01', '1991-06-11T22:00:00', att=36, depth=0, note='Buried in rocky fill, exact depth unknown: 0 m placeholder. Seismic lost June 11 22:00 local.')
    add('BUR', '1991-05-05', '1991-06-12', att=24, note='Link lost June 11; the epoch ends with that local day.')
    add('PPO', '1991-05-03', '1991-05-12', att=36, depth=0)
    add('PPO', '1991-05-12', '1991-05-30', sensor='L22-D', att=24, depth=0)
    add('PPO', '1991-05-30', '1991-06-12', sensor='L22-D', att=30, depth=0)
    for ch in ['EHN', 'EHE']:
        add('PPO', '1991-05-12', '1991-06-12', sensor='L22-D', att=30, ch=ch, depth=0)
    add('BUG', '1991-05-11', '1991-06-16', att=18, note='Lightning outage and May 28 reinstall in chronology; exact outage onset unknown. June 15 heat damage.')
    add('GRN', '1991-05-14', '1991-06-16', att=24, depth=0, note='24 dB inferred retrospectively from pulses. Reburied 1.5 m in early June; exact date unknown, initial depth unknown; 0 m placeholder.')
    add('CAB', '1991-05-01', '1991-05-05T12:45:00', att=30, depth=0, note='Buried sensor; depth unreported, 0 m placeholder.')
    add('CAB', '1991-05-05T12:45:00', '1991-06-16', att=24, depth=0, note='Unknown burial depth: 0 m placeholder. Attenuation change at 12:45 local. Battery outage separate. Later calibration continuity unverified.')
    headers = pd.read_csv(Path(header_data_dir or DEFAULT_HEADER_DATA) / 'waveform_header_presence.csv', dtype={'file_month': str})
    observed = observed_identity_bounds(headers)
    # The eruption climax: 15 June ends at local midnight, 15 June 16:00 UTC.
    climax_end = pht_to_utc('1991-06-16')
    documented = {(r['station'], r['channel']) for r in rows}
    # Post-eruption identities: those recorded in July/August (UTC), from their first surviving header after the climax.
    post = headers[headers.file_month.isin(['07', '08'])]
    for (sta, ch), group in post.groupby(['station', 'channel']):
        h = headers[(headers.station == sta) & (headers.channel == ch)]
        first = pd.to_datetime(h['first'], utc=True).dt.tz_localize(None)
        first = first[first >= pd.Timestamp(climax_end)].min()
        if (sta, ch) in documented:
            # CAB/BUG Z continue across the climax: the post-eruption epoch starts where the documentary one ends.
            start = climax_end
            note = 'Post-eruption configuration unknown. Starts where the documentary epoch ends (15 June, local midnight); not a documented field change.'
        else:
            start = first.floor('s').strftime('%Y-%m-%dT%H:%M:%S')
            note = 'Post-eruption configuration unknown. Starts at the first surviving header of this identity (UTC); not a verified installation date.'
        add(sta, start, sensor='unknown', ch=ch, att=None, depth=0, utc=True, basis='first_header',
            note=note + ' Depth 0 is an unknown-depth placeholder.')
    # Make every archived trace fall inside an epoch of its identity.
    for b in observed.itertuples(index=False):
        mine = sorted((r for r in rows if (r['station'], r['channel']) == (b.station, b.channel)),
                      key=lambda r: r['start'] or '')
        if not mine:
            continue
        first, last = b.first.floor('s'), b.last.ceil('s')
        head = mine[0]
        if head['start'] and first < pd.Timestamp(head['start']):
            head['note'] += (f' Start extended from {head["start"]} UTC to the first archived sample: '
                             'data with this label precede the documented installation.')
            head['start'] = first.strftime('%Y-%m-%dT%H:%M:%S')
        tail = mine[-1]
        if tail['end'] and last > pd.Timestamp(tail['end']):
            add(b.station, tail['end'], last.strftime('%Y-%m-%dT%H:%M:%S'), sensor='unknown', ch=b.channel, att=None,
                depth=tail['depth_m'], utc=True, basis='label_after_documented_end',
                note='Waveforms still carry this label after the documented end; the signal source is unverified '
                     '(the acquisition station table may not have been updated after the eruption). Ends at the last archived sample.')
        elif tail['end'] is None:
            continue
        # Data between consecutive epochs (a gap) would also be uncovered; none is expected.
    missing = uncovered_data(rows, headers)
    if len(missing):
        raise ValueError(f'{len(missing)} daily header rows fall outside every epoch: {missing.head().to_dict("records")}')
    add('PI2', None, sensor='unknown', ch='ELZ', depth=0, utc=True,
        note='Undated low-gain identity PI2L in station list and surviving PHA picks. Raw DMX daily samples contain only PI2Z; source acquisition/picking medium unresolved. No verified response or deployment epoch.')
    for sta, start, end in [('UBO', '1991-05-31', '1991-06-14'), ('PPO', '1991-06-02', '1991-06-12')]:
        for ch in ['WA1', 'WA2']:
            add(sta, start, end, sensor='electronic tilt, model unverified', ch=ch, depth=0, rate=1 / 600, note='Draft radial/tangential tilt codes, not geographic N/E. UBO axes 274/4 degrees; sign convention west-up/south-up needs raw-data verification. PPO floor installation unreliable; June 10 improvement incomplete. End boundaries approximate; no calibration supplied.')
    return rows


## Embedded StationXML and nominal-response construction

The provisional inventory records identities, documentary locations, epoch bounds, comments and known sensor descriptions. It deliberately has no complete digital-count response. Missing required elevation/depth fields use the documented placeholders/modern DEM estimate; inspect their source comments rather than treating them as surveyed measurements.

The separate nominal-voltage product models the assumed sensor poles/zeros and attenuation. Its output is volts, not ADC counts. Responses are built only for recognized L4-C/L22-D epochs; unknown post-eruption sensors remain uncalibrated. Sensor sensitivity, damping and coil/shunt assumptions are reference-data inputs, not fit results. Never use this voltage-only model as proof of recovered end-to-end response. `nominal_count_response` adds nominal preamplifier, VCO, discriminator and ADC stages from `nominal_electronics.json` (section 7); the result is a labelled model, not a calibration.

In [ ]:
def nominal_response(sensor, att):
    cfg = load('response_assumptions.json')['sensors'][sensor]
    w = 2 * np.pi * cfg['natural_frequency_hz']
    z = cfg['damping']
    f = 5.0
    poles = [complex(-z * w, w * np.sqrt(1 - z * z)), complex(-z * w, -w * np.sqrt(1 - z * z))]
    s = 2j * np.pi * f
    shape = abs(s * s / ((s - poles[0]) * (s - poles[1])))
    gain = cfg['sensitivity_v_per_mps'] * shape
    stages = [PolesZerosResponseStage(1, gain, f, 'M/S', 'V', 'LAPLACE (RADIANS/SECOND)', f, [0j, 0j], poles, normalization_factor=1 / shape)]
    if att is not None:
        stages.append(ResponseStage(2, 10 ** (-att / 20), f, 'V', 'V'))
    return Response(instrument_sensitivity=InstrumentSensitivity(gain * (10 ** (-att / 20) if att is not None else 1), f, 'M/S', 'V'), response_stages=stages)

def load_electronics():
    """Nominal electronics stages (10_conversion/nominal_electronics.json): assumptions, not calibrations."""
    return json.loads((BASE / 'nominal_electronics.json').read_text())


def nominal_count_response(sensor, att, electronics=None):
    """Nominal end-to-end response, ground velocity to digital counts.

    Stages: sensor poles/zeros (M/S->V); preamplifier with the station attenuator (V->V, gain at zero
    attenuation minus the documented attenuation); VCO (V->HZ); discriminator (HZ->V); ADC (V->COUNTS).
    Every electronics value is a labelled assumption from nominal_electronics.json. Filters after the
    preamplifier are not modelled (passband assumed flat at the reference frequency).
    """
    if att is None:
        raise ValueError('a count response needs the documented attenuation')
    e = (electronics or load_electronics())['stages']
    sensor_resp = nominal_response(sensor, None)
    s1 = sensor_resp.response_stages[0]
    f = s1.stage_gain_frequency
    pre_db = e['preamp_zero_attenuation_gain_db']['value'] - att
    # Analog stages as pole-zero stages without poles or zeros, so StationXML keeps their units and
    # names (ObsPy writes a bare gain stage without either); the ADC as a coefficients stage.
    analog = [(10 ** (pre_db / 20), 'V', 'V', f'J502 {pre_db:g}dB'),
              (e['vco_hz_per_v']['value'], 'V', 'HZ', 'VCO'),
              (e['discriminator_v_per_hz']['value'], 'HZ', 'V', 'J101/J110')]
    stages = [s1]
    total = sensor_resp.instrument_sensitivity.value
    for n, (g, iu, ou, name) in enumerate(analog, 2):
        stages.append(PolesZerosResponseStage(n, g, f, iu, ou, 'LAPLACE (RADIANS/SECOND)', f, [], [],
                                              normalization_factor=1.0, name=name))
        total *= g
    adc = e['adc_counts_per_v']['value']
    stages.append(CoefficientsTypeResponseStage(5, adc, f, 'V', 'COUNTS', 'DIGITAL', numerator=[1.0], denominator=[],
                                                decimation_input_sample_rate=SEISMIC_RATE_HZ, decimation_factor=1,
                                                decimation_offset=0, decimation_delay=0.0, decimation_correction=0.0,
                                                name='DT2821'))
    total *= adc
    return Response(instrument_sensitivity=InstrumentSensitivity(total, f, 'M/S', 'COUNTS'), response_stages=stages)


def inventory(rows, models=False):
    """models: False (no responses), True or 'voltage' (sensor and attenuator, output V), 'counts' (nominal end-to-end)."""
    models = 'voltage' if models is True else models
    electronics = load_electronics() if models == 'counts' else None
    stations = []
    for info in load('stations.json'):
        elev = info['elevation_m']
        if elev is None:
            import xarray as xr
            with xr.open_dataarray(DATA / 'modern_earth_relief.nc', engine='scipy') as grid:
                elev = float(grid.interp(lon=info['longitude'], lat=info['latitude']))
        comments = [Comment('DRAFT: historical datum unknown; plotting assumes geographic WGS84. ' + info['coordinate_source']), Comment('Alternative coordinates: ' + json.dumps(info['alternatives'])), Comment('Unknown electronics: VCO/preamp, discriminator/filter and ADC scale. No complete counts-to-ground-motion response supplied.')]
        if info['elevation_m'] is None:
            comments.append(Comment('Required elevation field estimated from MODERN 15-arcsec DEM; not a historical measurement.'))
        st = Station(info['code'], info['latitude'], info['longitude'], elev, site=Site(name=info['name']), comments=comments)
        for row in [r for r in rows if r['station'] == info['code']]:
            ch = row['channel']
            az = {'Z': 0, 'N': 0, 'E': 90}.get(ch[-1])
            dip = -90 if ch.endswith('Z') else 0
            if ch.startswith('WA'):
                az = {'WA1': 274, 'WA2': 4}[ch] if info['code'] == 'UBO' else None
            c = Channel(ch, '', info['latitude'], info['longitude'], elev, row['depth_m'], azimuth=az, dip=dip, sample_rate=row['sample_rate_hz'], start_date=UTCDateTime(row['start']) if row['start'] else None, end_date=UTCDateTime(row['end']) if row['end'] else None, comments=[Comment(row['note']), Comment('Attenuation dB: ' + str(row['attenuation_db']) + '; sensor: ' + row['sensor'] + '; electronics scale UNKNOWN')], sensor=Equipment(type=row['sensor'], description='Provisional historical identification; no serial-specific calibration.'))
            if models == 'voltage' and row['sensor'] in ['L4-C', 'L22-D']:
                c.response = nominal_response(row['sensor'], row['attenuation_db'])
                c.comments.append(Comment('NOMINAL VOLTAGE MODEL ONLY: output V, not digital counts. Sensor coil/shunt assumed; electronics omitted.'))
            if models == 'counts' and row['sensor'] in ['L4-C', 'L22-D'] and row['attenuation_db'] is not None:
                c.response = nominal_count_response(row['sensor'], row['attenuation_db'], electronics)
                e = electronics['stages']
                c.comments.append(Comment('NOMINAL COUNT MODEL, NOT A CALIBRATION. Sensor coil/shunt assumed (response_assumptions.json). '
                    f"Assumed electronics (nominal_electronics.json): preamplifier {e['preamp_zero_attenuation_gain_db']['value']:g} +/- {e['preamp_zero_attenuation_gain_db']['uncertainty_db']:g} dB at zero attenuation; "
                    f"VCO {e['vco_hz_per_v']['value']:g} Hz/V; discriminator {e['discriminator_v_per_hz']['value']:g} V/Hz; ADC {e['adc_counts_per_v']['value']:g} counts/V. "
                    'Discriminator and anti-alias filters not modelled.'))
            st.channels.append(c)
        stations.append(st)
    return Inventory([Network('XB', stations=stations, description='Provisional 1991 Pinatubo network, including undated and auxiliary channels')], source='Pinatubo evidence reconstruction — DRAFT; unknown electronics explicit')

## Embedded maps and telemetry/operations diagrams

Maps use the local modern relief grid and the selected historical station-coordinate evidence. They show configuration/site identity, not daily uptime or a recovered 1991 summit surface. The telemetry diagram is schematic; node positions are layout coordinates, not geographical measurements. Historical routes and failed attempts are distinguished in captions rather than inferring connectivity from proximity.

The operations diagram distinguishes analog telemetry/helicorders from ADC acquisition and separate computer clocks. Product-transfer cadence does not imply sampling rate or clock synchronization. These functions save PNG/SVG figures and diagram-edge tables when called by the build, and close their figures afterward.

In [ ]:
def maps(out, notes=True):
    """Draw the network maps. notes=False omits the title and footnote and enlarges text (publication use; the caveats go in the caption)."""
    import xarray as xr
    from matplotlib.colors import LightSource
    with xr.open_dataarray(DATA / 'modern_earth_relief.nc', engine='scipy') as g:
        grid = g.load()
    sites = {s['code']: s for s in load('stations.json')}
    for period, codes in [('early_june', ['BUG', 'BUR', 'CAB', 'GRN', 'PIE', 'PPO', 'UBO']), ('july_august', ['BUG', 'CAB', 'CRW', 'DON', 'FNG', 'PI2', 'QAD'])]:
        fig, ax = plt.subplots(figsize=(9, 9))
        extent = [float(grid.lon.min()), float(grid.lon.max()), float(grid.lat.min()), float(grid.lat.max())]
        rgb = LightSource(315, 45).shade(grid.values, cmap=plt.get_cmap('gist_earth'), vmin=0, vmax=1800, vert_exag=0.3, blend_mode='soft')
        ax.imshow(rgb, extent=extent, origin='lower', alpha=0.8, interpolation='bilinear')
        ax.contour(grid.lon, grid.lat, grid.values, levels=[500, 1000], colors='k', alpha=0.2, linewidths=0.5)
        offsets = {'CAB': (8, -16), 'PI2': (8, 7), 'PIE': (8, 7), 'PPO': (-35, 9), 'UBO': (-35, -15), 'GRN': (-35, 0), 'BUG': (-35, -12)}
        for code in codes:
            s = sites[code]
            ax.scatter(s['longitude'], s['latitude'], marker='^', s=95, color='#b41e29', edgecolor='white', zorder=4)
            ax.annotate(code, (s['longitude'], s['latitude']), xytext=offsets.get(code, (8, 8)), textcoords='offset points', fontweight='bold', fontsize=11 if notes else 14, bbox=dict(facecolor='white', alpha=0.75, edgecolor='none', pad=1))
        ax.scatter(120.35, 15.13, marker='*', s=160, color='black')
        ax.annotate('Pinatubo\n(approx.)', (120.35, 15.13), xytext=(10, 0) if notes else (14, -4), textcoords='offset points', fontsize=8 if notes else 11)
        x, y = (120.22, 15.015)
        dx = 5 / (111.32 * np.cos(np.deg2rad(y)))
        ax.plot([x, x + dx], [y, y], color='k', lw=3)
        ax.text(x + dx / 2, y + 0.006, '5 km', ha='center', fontsize=None if notes else 13)
        ax.set(xlim=(120.19, 120.6), ylim=(14.99, 15.44))
        ax.set_xlabel('Longitude °E', fontsize=None if notes else 14)
        ax.set_ylabel('Latitude °N', fontsize=None if notes else 14)
        if notes:
            ax.set_title('Pinatubo seismic network — ' + ('early June 1991' if period == 'early_june' else 'July–August 1991'))
        else:
            ax.tick_params(labelsize=12)
            ax.xaxis.set_major_locator(plt.MultipleLocator(0.1))
        ax.set_aspect(1 / np.cos(np.deg2rad(15.2)))
        ax.grid(alpha=0.15)
        if notes:
            fig.text(0.11, 0.035, 'Modern 15-arcsec Earth relief: not the 1991 surface. Historical coordinate datum unresolved.\nPVO/repeater positions unverified and unplotted. ' + ('Outages and June 11 losses are in the chronology.' if period == 'early_june' else 'July/August union of header identities; continuous operation and deployment dates unverified.'), fontsize=8)
        fig.subplots_adjust(bottom=0.18 if notes else 0.08)
        from matplotlib.cm import ScalarMappable
        from matplotlib.colors import Normalize
        cb = fig.colorbar(ScalarMappable(norm=Normalize(0, 1800), cmap='gist_earth'), ax=ax, orientation='horizontal', fraction=0.035, pad=0.1, aspect=35)
        cb.set_label('Modern elevation (m); shading modifies colours', fontsize=8 if notes else 13)
        cb.ax.tick_params(labelsize=8 if notes else 12)
        tight = {} if notes else dict(bbox_inches='tight', pad_inches=0.05)
        fig.savefig(out / f'network_{period}.png', dpi=200, **tight)
        fig.savefig(out / f'network_{period}.svg', **tight)
        plt.close(fig)

def diagram(out, name, title, nodes, edges, footer, notes=True):
    """Draw a box-and-arrow diagram. notes=False omits the title and footer and enlarges text."""
    fig, ax = plt.subplots(figsize=(13, 8))
    ax.set(xlim=(0, 13), ylim=(0, 8))
    ax.axis('off')
    if notes:
        ax.set_title(title, fontsize=15, pad=16)
    half_w = 0.95 if notes else 1.12
    for key, (x, y, label) in nodes.items():
        ax.add_patch(FancyBboxPatch((x - half_w, y - 0.36), 2 * half_w, 0.72, boxstyle='round,pad=.06', facecolor='#e7f0f7', edgecolor='#315a73', zorder=2))
        ax.text(x, y, label, ha='center', va='center', fontsize=8 if notes else 9.5, zorder=3)
    records = []
    for a, b, label in edges:
        x, y, _ = nodes[a]
        xx, yy, _ = nodes[b]
        dx, dy = (xx - x, yy - y)
        t = min((half_w + 0.12) / abs(dx) if dx else float('inf'), 0.47 / abs(dy) if dy else float('inf'))
        ax.add_patch(FancyArrowPatch((x + t * dx, y + t * dy), (xx - t * dx, yy - t * dy), arrowstyle='-|>', mutation_scale=12, shrinkA=0, shrinkB=0, color='#567080', zorder=1))
        ax.text((x + xx) / 2, (y + yy) / 2 + 0.13, label, fontsize=7 if notes else 9, ha='center', bbox=dict(facecolor='white', edgecolor='none', alpha=0.9, pad=1))
        records.append(dict(source=a, target=b, link=label))
    if notes:
        fig.text(0.06, 0.025, footer, fontsize=8)
    fig.savefig(out / f'{name}.png', dpi=180, bbox_inches='tight')
    fig.savefig(out / f'{name}.svg', bbox_inches='tight')
    plt.close(fig)
    pd.DataFrame(records).to_csv(out / f'{name}_edges.csv', index=False)

def diagrams(out, notes=True):
    nodes = {'BUG': (1, 6.8, 'BUG'), 'GRN': (4, 6.8, 'GRN + repeater'), 'PIE': (4, 5.3, 'PIE'), 'UBO': (4, 4.1, 'UBO seismic + tilt\nseparate radio paths'), 'CAB': (4, 2.8, 'CAB'), 'BUR': (1, 1.1, 'BUR'), 'PPO': (4, 1.1, 'PPO Z/N/E + BUR\nLDR: tilt / rain'), 'BH': (8, 1.1, 'Bunker Hill\nradio / microwave'), 'PVO': (11, 4.1, 'PVO at Clark\nreceivers / discriminators')}
    edges = [('BUG', 'GRN', 'radio'), ('GRN', 'PVO', 'radio'), ('PIE', 'PVO', 'radio'), ('UBO', 'PVO', 'radio / LDR'), ('CAB', 'PVO', 'radio'), ('BUR', 'PPO', 'radio'), ('PPO', 'BH', 'multiplex radio'), ('BH', 'PVO', 'military microwave')]
    diagram(out, 'telemetry', 'Documented pre-eruption telemetry — schematic, not geographic', nodes, edges, 'Lockhart / Murray. Bunker Hill link lost June 11. June 14 Crow’s Nest attempt failed; not an operating route.\nRebuilt July/August routes are unresolved; no post-eruption links inferred from station coordinates.', notes=notes)
    nodes = {'analog': (1.2, 6.8, 'Radio discriminators\nanalog station signals'), 'drum': (4.1, 6.8, 'Helicorders\nseparate clock domain'), 'willie': (4.1, 5.3, 'WILLIE IBM-AT\nMDETECT + SSAM'), 'rick': (1.3, 3.9, 'RICKY IBM-XT\nlahar acquisition'), 'sleep': (1.3, 2.5, 'SLEEPY IBM-XT\ntilt acquisition'), 'rsam': (1.3, 1.1, 'Tandy Model 100\nRSAM acquisition'), 'share': (4.3, 2.5, 'Buffered serial\nprintsharer'), 'pork': (7.2, 2.5, 'PORKY IBM-AT\naggregate / graphics'), 'lan': (8, 5.3, '2 Mb/s peer LAN'), 'squab': (11.4, 6.8, 'SQUAB 386\nPCEQ / HYPO71PC / tape'), 'dave': (11.4, 5.3, 'DAVE / ANDY\nanalysis / communications'), 'remote': (11.4, 2.5, 'ANDY: 9600-baud modem\nRemote → CVO / AVO')}
    edges = [('analog', 'drum', 'analog'), ('analog', 'willie', '100 sps / 12-bit'), ('willie', 'share', 'SSAM: 1 min'), ('rick', 'share', 'when received'), ('sleep', 'share', '10 min'), ('rsam', 'share', '10 min'), ('share', 'pork', 'serial'), ('willie', 'lan', 'event waveforms'), ('pork', 'lan', 'shared products'), ('lan', 'squab', 'events'), ('lan', 'dave', 'files'), ('dave', 'remote', 'communications')]
    diagram(out, 'pvo_operations', 'PVO operations systems — acquisition clocks remain independent', nodes, edges, 'Murray: initial 1991 configuration; later software improvements are not assumed present from the start.\nAggregation and LAN transfer do not synchronize clocks. Cadences describe products/transfers, not all native sampling rates.', notes=notes)

## Embedded build and audit functions

`build_all()` writes the source-specific station/epoch tables, coordinate alternatives, chronology, timing notes, provisional/voltage XML, maps, diagrams and draft validation summary to the explicit external output directory. It consumes the selected header-scan folder rather than implicitly choosing an older archive.

`scan_converted_headers()` is the active scanner: it scans the configured WAV root and selected years, records files/failures and optionally hashes samples. It supersedes the old `scan_headers.py` helper, whose reconstructed DATA_TOP/FAIR path and fixed June–August selection could inspect the wrong archive. That obsolete scanner is removed rather than retaining two divergent implementations.

`audit_inventory()` checks schemas, same-ID epoch overlap, undated identities and response units. `exclusive_boundary_copy()` writes a separate endpoint-selection copy; its one-microsecond adjustment is a technical convention, not measured time accuracy. `package_evidence()` copies reference metadata, hashes source/configuration/notebook files and records that no clock correction or complete count response was supplied. These are definitions only; execution remains in the numbered workflow cells below.

In [ ]:
def build_all(output_dir=None, *, header_data_dir=None):
    project = BASE.parent
    if str(project) not in sys.path:
        sys.path.insert(0, str(project))
    from pinatubo_config import external_output_root
    if output_dir is None:
        from lib.config import pipeline_paths
        output_dir = pipeline_paths()['NETWORK_HISTORY_OUTPUT']
    out = external_output_root(output_dir, relative_to=BASE)
    out.mkdir(parents=True, exist_ok=True)
    rows = build_channel_epochs(header_data_dir=header_data_dir)
    pd.DataFrame(rows).to_csv(out / 'pinatubo_channel_epochs.csv', index=False)
    stationdata = load('stations.json')
    pd.DataFrame([{k: v for k, v in s.items() if k != 'alternatives'} for s in stationdata]).to_csv(out / 'pinatubo_stations.csv', index=False)
    from pyproj import Geod
    geod = Geod(ellps='WGS84')
    alternatives = [dict(station=s['code'], **a, horizontal_distance_from_selected_m=geod.inv(s['longitude'], s['latitude'], a['longitude'], a['latitude'])[2], elevation_difference_from_selected_m=a['elevation_m'] - s['elevation_m'] if a['elevation_m'] is not None and s['elevation_m'] is not None else None) for s in stationdata for a in s['alternatives']]
    pd.DataFrame(alternatives).to_csv(out / 'alternative_coordinates.csv', index=False)
    hist = load('history_evidence.json')

    def chronological(r):
        date = r['date_as_reported']
        match = re.search('1991-\\d\\d(?:-\\d\\d)?', date)
        return match.group(0) if match else {'early June 1991': '1991-06-01', 'latter May and early June 1991': '1991-05-20', 'early July 1991': '1991-07-01', 'early April 1991': '1991-04-01'}.get(date, '9999')
    hist.sort(key=chronological)
    pd.DataFrame(hist).to_csv(out / 'pinatubo_deployment_history.csv', index=False)
    clocks = load('timing_evidence.json')
    pd.DataFrame(clocks).to_csv(out / 'timing_evidence.csv', index=False)
    inventory(rows).write(str(out / 'pinatubo_provisional.xml'), format='STATIONXML', validate=True)
    inventory(rows, True).write(str(out / 'pinatubo_nominal_voltage_models.xml'), format='STATIONXML', validate=True)
    inventory(rows, 'counts').write(str(out / 'pinatubo_nominal_count_models.xml'), format='STATIONXML', validate=True)
    text = ['# Pinatubo network chronology — evidence draft', 'Every change currently documented in the reviewed sources is retained below. Absence of an entry is not evidence of no change. Date precision and source clock basis must be preserved; coordinate discrepancies do not establish relocations.', '']
    for r in hist:
        text.append(f"- **{r['date_as_reported']} — {r['station']}: {r['event']}**. {r['detail']} Precision: {r.get('date_precision', 'unspecified')}. Source: {r.get('source', 'unspecified')}.")
    text += ['', '## Archive observations', 'July/August headers contain BUG, CAB, CRW, DON, FNG, PI2 and QAD; CAB includes N and Z. These establish recorded identities, not installation dates or valid signals. Old BUR/GRN/PIE/PPO/UBO names persist in some late-June headers after documented failures. Stale configuration, placeholders and timing errors remain hypotheses requiring waveform inspection. Explicit current mapping preserves PI2L as ELZ, but sampled raw DMX records contain PI2Z only. Surviving monthly PHA records include PI2L arrivals; the acquisition/picking medium and missing low-gain waveforms remain unresolved.', '', 'Rain/lahar acquisition and remote platform coordinates, calibration and installation dates remain incomplete. BOB/STATIONS.TXT distinguishes LSAC, USAC, MARL and ABAC; these are not invented as georeferenced stations.']
    text += ['', 'Primary references: [Lockhart](https://pubs.usgs.gov/pinatubo/lock/), [Murray](https://pubs.usgs.gov/pinatubo/murray/), [Ewert](https://pubs.usgs.gov/pinatubo/ewert/), [Bautista](https://pubs.usgs.gov/pinatubo/bbautist/). Source files and hashes: ../data/sources.json.']
    (out / 'chronology.md').write_text('\n\n'.join(text))
    header = pd.read_csv(Path(header_data_dir or DEFAULT_HEADER_DATA) / 'waveform_header_presence.csv', dtype={'file_month': str})
    observed = header[header.file_month.isin(['07', '08'])].groupby(['station', 'channel']).agg(first_july_august_header=('first', 'min'), last_july_august_header=('last', 'max'), files=('files', 'sum')).reset_index()
    observed.to_csv(out / 'rebuilt_header_identity_bounds.csv', index=False)
    timing = ['# Timing evidence and correction limits', 'Waveform times (step 10) and these channel epochs are on UTC; see 10_conversion/TIMING_AND_CLOCKS.md. Field-note dates are Philippine local time, converted to UTC. Daily WWV checks were intended, but checks were sporadic; nominal ±0.1 s setting accuracy is not an archive accuracy claim.']
    for r in clocks:
        timing.append('## ' + r['system'])
        timing.extend(('**' + key.replace('_', ' ').capitalize() + ':** ' + str(value) for key, value in r.items() if key != 'system'))
    (out / 'timing_issues.md').write_text('\n\n'.join(timing))
    maps(out)
    diagrams(out)
    inv = read_inventory(str(out / 'pinatubo_provisional.xml'))
    report = {'stations': sum((len(n.stations) for n in inv)), 'channel_epochs': sum((len(s.channels) for n in inv for s in n)), 'full_count_responses': sum((c.response is not None for n in inv for s in n for c in s)), 'status': 'DRAFT: unknown electronics, undated PI2 low-gain channel, approximate epoch bounds and provisional modern DEM LIL elevation explicit'}
    (out / 'validation_summary.json').write_text(json.dumps(report, indent=2))
    return report

from pathlib import Path

from collections import defaultdict

import hashlib, json, shutil

import pandas as pd

from obspy import read, read_inventory

from obspy.io.stationxml.core import validate_stationxml

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def scan_converted_headers(wav_root, outdir, *, years=(1991,), hash_waveforms=False):
    """Scan the exact converted WAV database, never a reconstructed DATA_TOP path."""
    root = Path(wav_root).expanduser().resolve()
    out = Path(outdir)
    out.mkdir(parents=True, exist_ok=True)
    if not root.is_dir():
        raise FileNotFoundError(root)
    rows = defaultdict(lambda: dict(files=0, first=None, last=None, sample_rates=set(), example_file=None))
    errors = []
    manifest = []
    files = sorted((p for year in years for p in (root / str(year)).rglob('*') if p.is_file() and (not p.name.startswith('.'))))
    if not files:
        raise FileNotFoundError(f'No converted files for {years} in {root}')
    for i, path in enumerate(files, 1):
        stat = path.stat()
        entry = {'path': str(path), 'bytes': stat.st_size, 'mtime_ns': stat.st_mtime_ns}
        if hash_waveforms:
            entry['sha256'] = sha256(path)
        try:
            st = read(str(path), headonly=True)
            seen = set()
            if not st:
                raise ValueError('No waveform traces')
            for tr in st:
                s = tr.stats
                month = path.relative_to(root).parts[1]
                key = (month, str(s.starttime.date), s.network, s.station, s.location, s.channel)
                r = rows[key]
                if key not in seen:
                    r['files'] += 1
                    seen.add(key)
                start, end = (str(s.starttime), str(s.endtime))
                r['first'] = min(r['first'], start) if r['first'] else start
                r['last'] = max(r['last'], end) if r['last'] else end
                r['sample_rates'].add(float(s.sampling_rate))
                r['example_file'] = r['example_file'] or str(path)
            entry.update(status='read', traces=len(st))
        except Exception as exc:
            entry.update(status='failed', error=str(exc))
            errors.append({'path': str(path), 'error': str(exc)})
        manifest.append(entry)
        if i % 1000 == 0:
            print(f'Headers inspected: {i}/{len(files)}', flush=True)
    records = [dict(file_month=k[0], date=k[1], network=k[2], station=k[3], location=k[4], channel=k[5], **{**r, 'sample_rates': ','.join(map(str, sorted(r['sample_rates'])))}) for k, r in rows.items()]
    fields = ['file_month', 'date', 'network', 'station', 'location', 'channel', 'files', 'first', 'last', 'sample_rates', 'example_file']
    frame = pd.DataFrame(records, columns=fields).sort_values(['date', 'station', 'channel'])
    frame.to_csv(out / 'waveform_header_presence.csv', index=False)
    pd.DataFrame(errors, columns=['path', 'error']).to_csv(out / 'header_scan_failures.csv', index=False)
    (out / 'waveform_file_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
    (out / 'header_scan_summary.json').write_text(json.dumps({'source_root': str(root), 'years': list(years), 'files_read': len(files), 'failures': len(errors), 'waveform_content_hashing': hash_waveforms, 'identity_only': True}, indent=2) + '\n')
    return frame

def audit_inventory(path):
    inv = read_inventory(str(path))
    schema, errors = validate_stationxml(str(path))
    overlaps = []
    undated = []
    responses = []
    for network in inv:
        for station in network:
            for i, a in enumerate(station):
                ident = f'{network.code}.{station.code}.{a.location_code}.{a.code}'
                if a.start_date is None:
                    undated.append(ident)
                if a.response:
                    responses.append({'id': ident, 'input_units': a.response.instrument_sensitivity.input_units, 'output_units': a.response.instrument_sensitivity.output_units})
                for b in station.channels[i + 1:]:
                    if (a.code, a.location_code) != (b.code, b.location_code):
                        continue
                    starts = [t for t in (a.start_date, b.start_date) if t is not None]
                    ends = [t for t in (a.end_date, b.end_date) if t is not None]
                    lo = max(starts) if starts else None
                    hi = min(ends) if ends else None
                    if lo is None or hi is None or lo <= hi:
                        overlaps.append({'id': ident, 'start': str(lo), 'end': str(hi), 'boundary_only': lo is not None and hi is not None and (lo == hi)})
    return {'schema_valid': schema, 'schema_errors': list(errors), 'overlaps': overlaps, 'undated_channels': undated, 'responses': responses, 'stations': sum((len(n) for n in inv)), 'channel_epochs': sum((len(s) for n in inv for s in n))}

def exclusive_boundary_copy(source, destination):
    """Separate technical XML copy; shared end is 1 microsecond before next start.

    Source XML and documentary CSV are untouched. The epsilon encodes a selection
    convention, not measured temporal precision or a revised historical instant.
    """
    from obspy.core.inventory import Comment
    inv = read_inventory(str(source))
    changes = []
    for net in inv:
        for station in net:
            for a in station:
                for b in station:
                    if a is b or (a.code, a.location_code) != (b.code, b.location_code):
                        continue
                    if a.end_date is not None and b.start_date is not None and (a.end_date == b.start_date):
                        old = a.end_date
                        a.end_date = b.start_date - 1e-06
                        a.comments.append(Comment('ENDPOINT CONVENTION: end set 1 microsecond before next epoch start for unambiguous selection; not observed time precision. Historical date/clock uncertainty remains.'))
                        changes.append({'id': f'{net.code}.{station.code}.{a.location_code}.{a.code}', 'documentary_end': str(old), 'encoded_end': str(a.end_date)})
    inv.write(str(destination), format='STATIONXML', validate=True)
    return changes

def package_evidence(data_dir, out, config_file, *, code_files=(), canonical_files=None):
    data = Path(data_dir)
    out = Path(out)
    target = out / 'source_metadata'
    target.mkdir(exist_ok=True)
    inputs = {}
    for p in sorted(data.iterdir()):
        if p.is_file() and p.suffix.lower() in ('.json', '.csv', '.txt', '.dat', '.inp'):
            shutil.copy2(p, target / p.name)
            inputs[p.name] = sha256(p)
    canonical_inputs = {}
    for name, source in (CANONICAL_REFERENCE_FILES if canonical_files is None else canonical_files).items():
        source = Path(source)
        if not source.is_file():
            raise FileNotFoundError(f'Missing canonical legacy station metadata: {source}')
        digest = sha256(source)
        canonical_inputs[name] = {'path': str(source.resolve()), 'sha256': digest}
        inputs['canonical_legacy:' + name] = digest
    (target / 'canonical_legacy_metadata.json').write_text(json.dumps(canonical_inputs, indent=2) + '\n')
    inputs['config:' + str(config_file)] = sha256(config_file)
    manifest = {'source_metadata_sha256': inputs, 'canonical_legacy_metadata': canonical_inputs, 'code_sha256': {str(p): sha256(p) for p in code_files}, 'clock_correction_applied': 'epochs on UTC: field-note dates converted from Philippine local time; archive times from step 10 (clock_regimes.csv)', 'complete_count_response_verified': False}
    (out / 'step70_provenance.json').write_text(json.dumps(manifest, indent=2) + '\n')
    p = out / 'chronology.md'
    if p.exists():
        p.write_text(p.read_text().replace('../data/sources.json', 'source_metadata/sources.json'))
    return manifest

## 2. Refresh observed channel identity and coverage

Read the converted files' headers for the selected years. Default `YEARS=(1991,)` covers the conversion study; extend explicitly for additional years. Each daily row is grouped by directory month, recorded date and SEED ID. A header proves a saved identity, not valid signal, successful installation or continuous uptime. Header-only reading does not detect dead channels or clipping. Recorded timestamps remain uncorrected.

The scan saves its exact source root, per-file path/size/modification-time inventory, failures and daily identity table. `HASH_WAVEFORMS=True` additionally reads and hashes every waveform's bytes (slower); otherwise the file manifest is a structural inventory, not proof of byte identity. Generated metadata and code are hashed later regardless.

**Safe cache reuse:** when `RUN_HEADER_SCAN=False`, the following block checks the recorded root, selected years and every listed file's size/mtime, and checks for newly added or missing files. It refuses stale evidence instead of silently falling back to the bundled historical scan. Historical first/last bounds must be rerun after changes to step 10.


In [ ]:
YEARS=(1991,)
RUN_HEADER_SCAN=True
HASH_WAVEFORMS=False
if RUN_HEADER_SCAN:
    headers=scan_converted_headers(WAV_ROOT,HEADER_OUTPUT,years=YEARS,hash_waveforms=HASH_WAVEFORMS)
else:
    summary=json.loads((HEADER_OUTPUT/'header_scan_summary.json').read_text())
    if Path(summary['source_root']).resolve()!=WAV_ROOT.resolve() or summary['years']!=list(YEARS):
        raise ValueError('Cached scan describes a different archive/year selection; rescan')
    previous=json.loads((HEADER_OUTPUT/'waveform_file_manifest.json').read_text())
    current={str(p.resolve()):p for year in YEARS for p in (WAV_ROOT/str(year)).rglob('*') if p.is_file() and not p.name.startswith('.')}
    if set(current)!={str(Path(r['path']).resolve()) for r in previous}:raise ValueError('Archive membership changed; rescan')
    for r in previous:
        f=current[str(Path(r['path']).resolve())];st=f.stat()
        if st.st_size!=r['bytes'] or st.st_mtime_ns!=r['mtime_ns']:raise ValueError('Archive file changed; rescan: '+str(f))
        if r.get('sha256') and sha256(f)!=r['sha256']:raise ValueError('Waveform hash changed; rescan')
    headers=pd.read_csv(HEADER_OUTPUT/'waveform_header_presence.csv',dtype={'file_month':str})
display(pd.DataFrame([json.loads((HEADER_OUTPUT/'header_scan_summary.json').read_text())]))
display(headers.head(20))
display(pd.read_csv(HEADER_OUTPUT/'header_scan_failures.csv'))


## 3. Construct the documentary reconstruction using the refreshed evidence

`build_all` writes history, coordinates, epoch tables, maps and two response variants. Passing `header_data_dir` explicitly makes this run use the scan above. Field-note dates are not silently replaced by waveform timestamps. Current post-eruption inference is still provisional and historically focused on June–August; the full-year scan supports additional inspection without claiming that every channel's deployment history has been solved.

Outputs include `pinatubo_provisional.xml` (identity/location/epoch metadata, no end-to-end responses) and `pinatubo_nominal_voltage_models.xml` (nominal sensor/attenuation models with **V**, not counts, as output) and `pinatubo_nominal_count_models.xml` (nominal end-to-end models with **COUNTS** as output; assumed electronics, section 7). Do not run `remove_response` on digital counts using the latter as if it were the complete acquisition response.


In [ ]:
report=build_all(OUTPUT,header_data_dir=HEADER_OUTPUT)
display(report)
stations=pd.read_csv(OUTPUT/'pinatubo_stations.csv')
epochs=pd.read_csv(OUTPUT/'pinatubo_channel_epochs.csv')
display(stations)
display(epochs)
display(Markdown((OUTPUT/'chronology.md').read_text()))


## 4. Review primary sources and unresolved sensor conflicts

**Working precedence for the pre-eruption network:** Lockhart et al. (1996), Appendix 1, is a field-note synopsis and is the primary working source. Keep conflicting assertions as source evidence, not silently overwritten values. Lockhart identifies L-4, which does not establish deployed coil/shunt resistance or serial calibration; the current L4-C numerical model remains a nominal assumption.

| Site | Lockhart Appendix 1 | Helicorder README_Metadata.txt (20 March 2025) and PIN_DRUM2/readme.txt (3 April 2025) |
|---|---|---|
| PIE | L-4; installed 30 April, 18 dB | L-1, vertical, 1 Hz |
| UBO | L-4; installed 1 May, 36 dB | L-1, vertical, 1 Hz |
| CAB | L-4; installed 1 May; attenuation 30→24 dB on 5 May | L-22, three-component, 2 Hz |

**Questions for John Power, Andy Lockhart, Tom Murray and other VDAP colleagues (not sent):** Are the README sensor descriptions errors, later replacements, or different recording paths? Was CAB changed around 12 June, when its scanned drum holdings begin? Which serials, coils, shunts and damping settings were deployed? Do the original notebooks/calibration records resolve this?

The next block displays alternative coordinate differences and exports the conflict register. CAB differs by about 1.25 km between some sources and BUR by about 279 m; these are not proof of relocation. The selected pre-eruption table supplied by Wendy still needs its final bibliographic citation. LIL's XML elevation is interpolated from modern relief only because the required value is absent historically.


In [ ]:
conflicts=pd.DataFrame([
    {'station':'PIE','primary_sensor':'L-4','other_sensor':'L-1, 1 Hz','primary_source':'Lockhart Appendix 1','other_source':'Helicorder READMEs, March/April 2025','status':'unresolved'},
    {'station':'UBO','primary_sensor':'L-4','other_sensor':'L-1, 1 Hz','primary_source':'Lockhart Appendix 1','other_source':'Helicorder READMEs, March/April 2025','status':'unresolved'},
    {'station':'CAB','primary_sensor':'L-4','other_sensor':'L-22, 3-component, 2 Hz','primary_source':'Lockhart Appendix 1','other_source':'Helicorder READMEs, March/April 2025','status':'unresolved'}])
conflicts.to_csv(OUTPUT/'sensor_conflict_register.csv',index=False)
display(conflicts)
display(pd.read_csv(OUTPUT/'alternative_coordinates.csv'))


## 5. Check both StationXML schemas and all channel epoch overlaps

This audit checks every repeated network/station/location/channel ID, not just PPO. Shared endpoints are expected where one epoch ends as the next begins: CAB on 5 May (04:45 UTC) and 15 June 16:00 UTC; PPO on 11 and 29 May 16:00 UTC; BUG on 15 June 16:00 UTC; and each `label_after_documented_end` epoch where its documentary epoch ends. A new scan can change this list, so inspect the actual audit below.

Lockhart resolves the physical meaning of the first three: CAB attenuation 30→24 dB (5 May, 12:45 local); PPO replacement L-4/36 dB→L-22D/Z24 dB (12 May); PPO Z24→30 dB (30 May). PPO N/E remain 30 dB from 12 May. Date-only entries are local midnight converted to UTC, not exact instants. The 15 June 16:00 UTC CAB/BUG transitions are reconstruction conventions, not documented field changes.

**Boundary policy:** documentary CSV keeps its reported/conventional bounds. An additional XML copy below encodes an end one microsecond before the next start where endpoints coincide. This is a selection convention, not microsecond scientific accuracy. Positive-duration overlaps remain errors requiring investigation. Undated PI2 low-gain entries are also listed: an open epoch can match any date even though its true dates are unknown.


In [ ]:
audits={name:audit_inventory(OUTPUT/name) for name in ['pinatubo_provisional.xml','pinatubo_nominal_voltage_models.xml','pinatubo_nominal_count_models.xml']}
for name,audit in audits.items():
    print(name,'schema valid:',audit['schema_valid'],'channels:',audit['channel_epochs'])
    display(pd.DataFrame(audit['overlaps']))
    print('Undated channels:',audit['undated_channels'])
    display(pd.DataFrame(audit['responses']))
(OUTPUT/'stationxml_audit.json').write_text(json.dumps(audits,indent=2))


In [ ]:
MAKE_SELECTION_SAFE_COPIES=True
boundary_changes={}
if MAKE_SELECTION_SAFE_COPIES:
    for name in audits:
        destination=OUTPUT/(Path(name).stem+'_selection_safe.xml')
        changes=exclusive_boundary_copy(OUTPUT/name,destination)
        checked=audit_inventory(destination)
        boundary_changes[name]={'changes':changes,'audit':checked}
        display(pd.DataFrame(changes))
        if checked['overlaps']:print('REVIEW REQUIRED: remaining overlaps',checked['overlaps'])
    (OUTPUT/'epoch_boundary_encoding.json').write_text(json.dumps(boundary_changes,indent=2))


## 6. PI2 normal and low gain: evidence versus observation

The 27 June and 8 July 1991 SEIS_MAP tables explicitly pair PI2Z (**NORMAL GAIN**) and PI2L (**LOW GAIN**). VCO frequencies are 1700 and 2720, respectively, with indicated shared radio frequency 411.825; units are contextual interpretations. The 8 July table gives 36 dB attenuation for PI2Z and `?` for PI2L. Do not substitute a generic post-1991 dual-VCO gain ratio.

STATIONS.INP lists both at 15°13.25′ N, 120°25.06′ E. PIN_STA_ALL gives site elevation 720 m. Under the audited ten-minute annual RSAM convention, both first have nonsentinel values at recorded 27 June 09:20; PI2L extends to 15 November 15:00 and PI2Z amplitude to 22 November 18:10. These are calendar labels, not independently established UTC.

### Acquisition evidence added 11 November 2026

| Product or check | Evidence | Status / limit |
|---|---|---|
| RSAM | Surviving PI2L and PI2Z files contain distinct amplitude series; the coverage above is based on nonsentinel samples. | PI2L recording confirmed. |
| Raw MDETECT event waveforms | Direct ObsPy DMX reads, bypassing FLOVOpy and all station-ID mapping, sampled the first file on each of 209 filename dates. The first files on 22 July and 6 October were unreadable; the next file on each date was readable. PI2 occurred in 159 daily samples, spanning 27 June–5 December 1991, always as `unk.PI2Z.._`; no sampled file contained PI2L. | Strong evidence of no explicitly labeled low-gain channel in the sampled event records; daily sampling is not an exhaustive search of all events and cannot exclude upstream mislabeling. |
| Conversion mapping | Explicit mapping checks preserve the distinction: `unk.PI2L.._` → `XB.PI2..ELZ`; `unk.PI2Z.._` → `XB.PI2..EHZ`. The inspected step-10 provenance contained 17,919 PI2 rows, all PI2Z → EHZ. | No evidence that the current mapper collapses an explicitly labeled PI2L channel into EHZ. |
| SSAM | Glenn Thompson reports checking the available SSAM data and finding no PI2L (chat correspondence, 11 November 2026). | User-reported archive inspection; this notebook does not independently repeat that SSAM check. |
| Paper helicorders | PI2L use has been suggested but has not been established from recorder labels or routing documentation. | Unconfirmed. |

**Updated interpretation after stages 20–34:** monthly PHA records explicitly contain PI2L arrivals (1,826 retained merged picks in the reviewed run). Thus the raw-DMX/SSAM absence does not establish that PI2L was never available for event picking. A different recording medium, unsampled/lost low-gain files or earlier processing remains possible. Step 32 retains original low-gain pick identity and supplies a separate PI2Z overlay target; that display policy is not a gain calibration or evidence of a surviving low-gain waveform. Ask colleagues which medium supplied the PI2L picks.

The reproducible raw-header sampling script and results are retained in the project review workspace under `diagnostics/pi2-gain-audit/`: `sample_daily_obspy_headers.py`, `daily-sample/daily_obspy_headers.csv`, `daily-sample/daily_obspy_headers.json`, and `daily-sample/unreadable_day_retries.json`. These are supporting review artifacts, not assumed pipeline outputs or notebook dependencies. Dates here follow raw filename/record labels and do not independently resolve clock offsets.

**Questions for VDAP colleagues:** what were the PI2 sensor, gain ratio, deployment dates, and channel routes into each computer and paper recorder? Was PI2L intentionally excluded from MDETECT and SSAM, and was it recorded on a helicorder? Could any upstream labels differ from the documented PI2Z/PI2L distinction? The next block checks this conversion run, without guessing a response or using lack of a waveform as proof the channel did not exist.


In [ ]:
pi2=headers[headers.station.astype(str).eq('PI2')]
display(pi2)
translation=Path(P['TRANSLATION_CSV'])
if translation.is_file():
    mapping=pd.read_csv(translation)
    mask=mapping.astype(str).apply(lambda column:column.str.contains('PI2',regex=False)).any(axis=1)
    display(mapping.loc[mask])
else:print('No step-10 trace translation table at',translation)
if pi2.empty or not pi2.channel.astype(str).eq('ELZ').any():
    print('No PI2 ELZ header in selected converted archive. Preserve documentary low-gain evidence; do not fabricate an observed epoch.')


## 7. Response chain and nominal count model

A digital count response requires **velocity → sensor → preamp/attenuation → VCO → discriminator/filter → ADC counts**. Only the sensor type and the station attenuation are documented. Step 70 writes three inventories:

| File | Responses |
|---|---|
| `pinatubo_provisional.xml` | none |
| `pinatubo_nominal_voltage_models.xml` | sensor and attenuator only, output **V** |
| `pinatubo_nominal_count_models.xml` | nominal end-to-end, output **COUNTS**, with every assumed electronics value from `10_conversion/nominal_electronics.json` named in a channel comment |

| Stage | Evidence | Nominal value used | Status |
|---|---|---|---|
| Sensor | Lockhart: L-4 (L-22D at PPO from 12 May) | L4-C 1 Hz, 170.9 V/(m/s) loaded; L-22D 2 Hz, 88 V/(m/s) | coil, shunt and serials not established; README sensor conflicts unresolved |
| Preamplifier | Lockhart: J502-based CVO boards, 18–36 dB attenuation by epoch; McChesney (1999): J502 unattenuated gain 90 dB | 90 dB at zero attenuation (±6 dB) | J502 value; the custom boards were not measured |
| VCO | McChesney (1999): J502/J512 deviation 115 Hz per 4.05 V | 28.4 Hz/V | documented for the J502 |
| Discriminator | Lockhart: J101 and J110; McChesney: VCO test-point and discriminator output have about the same amplitude | 4.05 V per 115 Hz (VCO × discriminator = 1 V/V); filter not modelled | scale inferred |
| ADC | Murray: 12-bit, DT2821 (Ellis, 1989); SUDS StationComp `atod gain=4` | ±10 V / 4 = ±2.5 V, 819.2 counts/V | board documented; range from the board specification and the gain field |
| End-to-end check | Lockhart: gains set for 100 mV p-p ambient noise at the VCO input | 82 counts p-p expected | compared below with recorded noise |

Post-eruption epochs have no documented sensor or attenuation and get no response. The count model is for exploration with explicit caveats (approximate ground-motion amplitudes, the wfdisc `calib` field); it is not a calibration and the response gate below stays `complete_count_response_verified: False`. Each stage takes a value documented for its equipment type, never unity by default.


In [ ]:
response_assumptions=json.loads((DATA/'response_assumptions.json').read_text())
display(response_assumptions)
electronics=load_electronics()
display(electronics)
response_gate={'complete_count_response_verified':False,
    'nominal_count_model':'pinatubo_nominal_count_models.xml: documented pre-eruption epochs only; electronics values are the assumptions in 10_conversion/nominal_electronics.json',
    'unverified':['preamp zero-attenuation gain (J502 value, 90 dB, assumed for the CVO boards)','discriminator scale (matched to the VCO deviation, assumed)','ADC full-scale range (DT2821 specification and SUDS atod gain field)','discriminator and anti-alias filters (not modelled)','sensor configuration/epoch conflicts'],
    'usable_now':'Provisional identity/location inventory; nominal voltage and count models for explicitly qualified exploration',
    'prohibited_claim':'These files establish calibrated counts-to-ground-motion response for the archive'}
(OUTPUT/'response_completeness.json').write_text(json.dumps(response_gate,indent=2))
display(response_gate)

# Lockhart: gains were set for 100 mV p-p ambient noise at the VCO input. Expected counts p-p from the
# nominal VCO, discriminator and ADC; compare with the first 5 s (pre-trigger) of records in the first
# three days of each documented epoch.
e=electronics['stages']
expected_pp=0.1*e['vco_hz_per_v']['value']*e['discriminator_v_per_hz']['value']*e['adc_counts_per_v']['value']
from obspy import read
wav=pd.read_csv(P['WAVEFORM_INDEX'],usecols=['wav_file','starttime','stations','status'])
wav=wav[wav.status.eq('written')].assign(t=lambda d:pd.to_datetime(d.starttime,format='mixed'))
noise_rows=[]
for r in pd.read_csv(OUTPUT/'pinatubo_channel_epochs.csv').itertuples():
    if r.sensor not in ('L4-C','L22-D') or pd.isna(r.attenuation_db):continue
    t0=pd.Timestamp(r.start).tz_localize(None);t1=min(pd.Timestamp(r.end).tz_localize(None),t0+pd.Timedelta(days=3))
    files=wav[(wav.t>=t0)&(wav.t<t1)&wav.stations.str.contains(r.station)].wav_file.head(60)
    pp=[]
    for f in files:
        try:st=read(f,format='MSEED').select(station=r.station,channel=r.channel)
        except Exception:continue
        for tr in st:
            x=tr.data[:int(5*tr.stats.sampling_rate)].astype(float)
            if len(x)>100 and np.ptp(x)<4000:pp.append(np.ptp(x-x.mean()))
    noise_rows.append(dict(station=r.station,channel=r.channel,start=r.start,attenuation_db=r.attenuation_db,records=len(pp),
        median_pp_counts=float(np.median(pp)) if pp else None,expected_pp_counts=expected_pp))
noise_check=pd.DataFrame(noise_rows).assign(ratio=lambda d:d.median_pp_counts/d.expected_pp_counts)
noise_check.to_csv(OUTPUT/'response_noise_check.csv',index=False)
display(noise_check.round(2))
print(f'expected {expected_pp:.0f} counts p-p; median observed/expected {noise_check.ratio.median():.2f} '
      f'(range {noise_check.ratio.min():.2f}-{noise_check.ratio.max():.2f}) over {noise_check.ratio.notna().sum()} epochs; '
      f'correlation of observed p-p with attenuation {noise_check[["median_pp_counts","attenuation_db"]].corr().iloc[0,1]:.2f}')


## 8. Clock domains, later time shifts and colleague questions

Each acquisition computer had its own offset/drift/reset history; helicorders had independent timing. Shared LAN/collection by PORKY does not synchronize acquired observations. Stations sharing an acquisition clock generally have better relative timing, but the same product label alone does not prove common timing.

MDETECT initially tagged events PC+8 h while SSAM used PC time; the early PC clock was local, and a later GMT transition was reported. Do not infer one correction for all epochs. Philippine local time is UTC+8; CVO Pacific civil time is UTC−8 in standard time and UTC−7 in daylight time (Philippines is 16 or 15 hours ahead), but that is not evidence a PC was set to CVO time.

| Product | Additional recovered evidence and action |
|---|---|
| TILT | TIME.TXT / BOB_UBO_.TXT state GMT→local conversion on 18 November 1991; PPO says 48 ten-minute positions. Do not shift again without verifying phase/day convention. |
| RAIN | Header says GMT→local, but CONVERT.FOR subtracts eight hours and does not update printed dates across midnight. Preserve labels and identify actual converter revision. |
| MUDFLOW | Many daily files state conversion to local time; others lack the header. Track per-file convention. |
| COUNTS/COSPEC | CAB counts and COSPEC explicitly say GMT; PIE counts do not. Midnight COSPEC tags do not establish measurement time. |
| Helicorders | README specifies Luzon local time; recorder-specific drift/reset history remains unknown. |

**Ask VDAP:** which board/ranges/gains and discriminator assignments were used? Can Andy/John/Tom clarify sensor conflicts, PI2L gain and event acquisition, exact gain-change epochs, local/GMT corrections, and available calibration sheets/pulses? This notebook records questions; it does not send correspondence or resolve them by assumption.


In [ ]:
display(Markdown((OUTPUT/'timing_issues.md').read_text()))
# Preserve the current notebook as part of the evidence so these additional notes travel with outputs.
notebook_source=PIPELINE/'70_network_history_and_stationxml.ipynb'
if notebook_source.is_file():shutil.copy2(notebook_source,OUTPUT/'70_network_history_and_stationxml.ipynb')


## 9. Inspect figures and package provenance

The map uses modern relief rather than the 1991 summit surface. Maps describe configurations/site identities, not daily uptime. Sensor and radio site coordinates require separate evidence; unverified relay geometry is not invented. The operations diagram distinguishes analog drums from ADC acquisition and independent clocks. Inspect label placement and interpret transfer cadences separately from native sampling.

The final packaging block copies source metadata (including the previously missing source index), hashes inputs/configuration/code, saves a validation manifest, and repairs the chronology's output-relative source link. It also hashes the generated scan manifest. Waveform hashes are present only if requested in block 2. Reading references with local Mac mini PDF paths does not prove those files are available on this machine; their stored hashes and public URLs retain their citation identity.


In [ ]:
for name in ['network_early_june','network_july_august','telemetry','pvo_operations']:
    display(Image(filename=str(OUTPUT/(name+'.png'))))


In [ ]:
provenance=package_evidence(DATA,OUTPUT,Path(P['CONFIG_FILE']),code_files=[
    PIPELINE/'70_network_history_and_stationxml.ipynb',PIPELINE/'nominal_electronics.json'])
provenance.update(created_utc=datetime.now(timezone.utc).isoformat(),waveform_root=str(WAV_ROOT.resolve()),
    header_file_manifest_sha256=sha256(HEADER_OUTPUT/'waveform_file_manifest.json'),
    header_summary=json.loads((HEADER_OUTPUT/'header_scan_summary.json').read_text()),
    stationxml_audits=audits,boundary_policy='Documentary original retained; separate selection-safe copy uses 1-microsecond endpoint encoding',
    response_completeness=response_gate)
try:
    import subprocess
    provenance['git_revision']=subprocess.check_output(['git','rev-parse','HEAD'],cwd=PIPELINE,text=True).strip()
    provenance['git_dirty']=bool(subprocess.check_output(['git','status','--porcelain'],cwd=PIPELINE,text=True).strip())
except Exception as exc:provenance['git_status_unavailable']=str(exc)
provenance['step10_metadata_sha256']={key:sha256(P[key]) for key in ['WAVEFORM_INDEX','TRANSLATION_CSV'] if Path(P[key]).is_file()}
(OUTPUT/'step70_provenance.json').write_text(json.dumps(provenance,indent=2))
print('Review package:',OUTPUT)
print('Schema checks:',{name:a['schema_valid'] for name,a in audits.items()})
print('Unverified response stages:',response_gate['unverified'])


## Completion criteria and next review

- Header failures are inspected, not silently treated as missing stations.
- Sensor conflicts and unknown coordinates/epochs remain visible.
- No unexplained positive-duration same-ID overlaps are accepted.
- Undated auxiliary/low-gain identities are not treated as verified all-time deployments.
- Response output units and missing stages are explicit.
- Original source files and raw clocks are unchanged; partial waveform recoveries require their step-10 provenance.
- The next outputs to review are waveform availability, individual/monthly picks, associations and hypocenters. These metadata products do not validate those downstream results by themselves.

**Primary references:** [Lockhart et al.](https://pubs.usgs.gov/pinatubo/lock/), [Murray et al.](https://pubs.usgs.gov/pinatubo/murray/), [Ewert et al.](https://pubs.usgs.gov/pinatubo/ewert/); additional BOB and helicorder files are identified above. Do not apply assumptions from later dual-VCO designs retrospectively to the 1991 chain.
